# Semantic Search RAG vs. Moment RAG

**Module 3 (Agentic RAG) assignment.** We build two retrieval-augmented pipelines over the
transcript of a single YouTube video and compare them:

- **Part 1 — Baseline semantic-search RAG:** fixed-size chunks → embeddings → cosine top-k → answer.
- **Part 2 — Moment RAG:** timestamped *moments* → ingestion enrichment (HyDE-style hypothetical
  questions) → query **decomposition** → **hybrid** retrieval (dense + BM25, RRF-fused) →
  **cross-encoder re-rank** → moment-level rollup with **click-to-timestamp citations**.
- **Part 3 — Comparison & analysis.**

The Moment RAG design mirrors the `Moment_RAG/` reference codebase from this module — same ideas
(query-side intelligence, HyDE-at-ingestion, RRF fusion, cross-encoder rerank, moment rollup),
on a lighter, fully transparent stack (numpy + `rank-bm25` + a small cross-encoder) instead of
the Qdrant/fastembed setup, so every step is readable here.

> **Video:** *How does a Vector Database work?* — https://www.youtube.com/watch?v=VVNYQKDLY5s


## Section 0 — Setup & transcript

We load the OpenAI client, then fetch the video's captions and reshape them into timed **cues**
(`{start_ms, end_ms, text}`) — the same shape the reference codebase uses. The timestamp on each
cue is what later lets a retrieved *moment* deep-link back into the video.

In [1]:
import os, json, re, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()  # reads OPENAI_API_KEY from the environment / .env

VIDEO_ID    = "VVNYQKDLY5s"
EMBED_MODEL = "text-embedding-3-small"
LLM_MODEL   = "gpt-4o-mini"

DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True)
ENC = tiktoken.get_encoding("cl100k_base")
print("setup ok")

setup ok


In [2]:
def fetch_transcript(video_id: str) -> list[dict]:
    """Fetch captions and reshape to timed cues. Cached to data/transcript.json.

    Handles both the legacy static API (youtube-transcript-api <=0.6.x) and the
    newer instance API (>=1.0).
    """
    cache = DATA_DIR / "transcript.json"
    if cache.exists():
        return json.loads(cache.read_text())

    from youtube_transcript_api import YouTubeTranscriptApi
    if hasattr(YouTubeTranscriptApi, "get_transcript"):          # <=0.6.x
        raw = YouTubeTranscriptApi.get_transcript(video_id)
        segs = [(r["text"], r["start"], r["duration"]) for r in raw]
    else:                                                         # >=1.0
        fetched = YouTubeTranscriptApi().fetch(video_id)
        segs = [(s.text, s.start, s.duration) for s in fetched]

    cues = []
    for text, start, dur in segs:
        text = (text or "").replace("\n", " ").strip()
        if not text:
            continue
        cues.append({
            "start_ms": int(start * 1000),
            "end_ms":   int((start + dur) * 1000),
            "text":     text,
        })
    cache.write_text(json.dumps(cues, indent=2))
    return cues

cues = fetch_transcript(VIDEO_ID)
print(f"{len(cues)} cues, ~{sum(len(ENC.encode(c['text'])) for c in cues)} tokens total")
cues[:3]

312 cues, ~2377 tokens total


[{'start_ms': 240,
  'end_ms': 4000,
  'text': "Let's say your company has an employee"},
 {'start_ms': 2000,
  'end_ms': 6160,
  'text': 'handbook that covers policies like time'},
 {'start_ms': 4000,
  'end_ms': 8160,
  'text': 'off requests, dress code, and equipment'}]

In [3]:
def embed_texts(texts: list[str], model: str = EMBED_MODEL, batch: int = 256) -> np.ndarray:
    """Embed a list of texts with OpenAI, batched, returned as a float32 matrix."""
    out = []
    for i in range(0, len(texts), batch):
        resp = client.embeddings.create(model=model, input=texts[i:i + batch])
        out.extend(d.embedding for d in resp.data)
    return np.array(out, dtype=np.float32)

def cosine_topk(qvec: np.ndarray, matrix: np.ndarray, k: int) -> list[tuple[int, float]]:
    """Top-k rows of `matrix` by cosine similarity to `qvec`."""
    if len(matrix) == 0:
        return []
    q = qvec / (np.linalg.norm(qvec) + 1e-9)
    M = matrix / (np.linalg.norm(matrix, axis=1, keepdims=True) + 1e-9)
    sims = M @ q
    idx = np.argsort(-sims)[:k]
    return [(int(i), float(sims[i])) for i in idx]

def fmt_ts(ms: int) -> str:
    s = int(ms / 1000); return f"{s // 60}:{s % 60:02d}"

def yt_link(ms: int) -> str:
    return f"https://www.youtube.com/watch?v={VIDEO_ID}&t={int(ms / 1000)}s"

print("helpers ready")

helpers ready


## Part 1 — Baseline semantic-search RAG

The classic pipeline: cut the transcript into **fixed-size token windows** (~256 tokens with 25%
overlap, mirroring the reference's `_window_chunks`), embed each, and retrieve the top-k by cosine
similarity. Chunk boundaries are arbitrary — they fall wherever the token budget runs out, not
where a topic actually starts or ends.

In [4]:
CHUNK_TOKENS = 256
OVERLAP = 0.25

def fixed_chunks(cues, chunk_tokens=CHUNK_TOKENS, overlap=OVERLAP):
    """Group consecutive cues into ~chunk_tokens windows with token overlap."""
    counts = [len(ENC.encode(c["text"])) for c in cues]
    stride_back = max(1, int(chunk_tokens * overlap))
    chunks, i, n = [], 0, len(cues)
    while i < n:
        tok, j = 0, i
        while j < n and tok + counts[j] <= chunk_tokens:
            tok += counts[j]; j += 1
        if j == i:
            j = i + 1
        window = cues[i:j]
        chunks.append({
            "chunk_id": f"c{len(chunks):04d}",
            "text": " ".join(c["text"] for c in window),
            "start_ms": window[0]["start_ms"],
            "end_ms": window[-1]["end_ms"],
            "n_tokens": tok,
        })
        if j >= n:
            break
        # step back a few cues so the next window overlaps by ~stride_back tokens
        back, t, k = 0, 0, j - 1
        while k >= i and t < stride_back:
            t += counts[k]; back += 1; k -= 1
        i = max(i + 1, j - back)
    return chunks

base_chunks = fixed_chunks(cues)
base_matrix = embed_texts([c["text"] for c in base_chunks])
print(f"{len(base_chunks)} fixed chunks, embedding matrix {base_matrix.shape}")

13 fixed chunks, embedding matrix (13, 1536)


In [5]:
def baseline_rag(query: str, k: int = 4) -> dict:
    qv = embed_texts([query])[0]
    hits = cosine_topk(qv, base_matrix, k)
    context = "\n\n".join(
        f"[{n+1}] {base_chunks[i]['text']}" for n, (i, _) in enumerate(hits)
    )
    prompt = (
        "Answer the question using ONLY the context below. "
        "If the answer isn't in the context, say so.\n\n"
        f"Context:\n{context}\n\nQuestion: {query}"
    )
    resp = client.chat.completions.create(
        model=LLM_MODEL, temperature=0.2,
        messages=[{"role": "user", "content": prompt}],
    )
    return {
        "answer": resp.choices[0].message.content,
        "chunks": [{"start": fmt_ts(base_chunks[i]["start_ms"]), "score": round(s, 3),
                    "text": base_chunks[i]["text"][:160] + "…"} for i, s in hits],
    }

demo = baseline_rag("How does a vector database find similar vectors?")
print(demo["answer"]); print("\n--- retrieved chunks ---")
for c in demo["chunks"]:
    print(f'  [{c["start"]}] score={c["score"]}  {c["text"]}')

The context does not provide a specific answer to how a vector database finds similar vectors.

--- retrieved chunks ---
  [3:54] score=0.649  are working with vector databases and this is the retrieval side. Meaning now that we store the meaning of those words, we have to take on the burden of the ret…
  [1:31] score=0.606  actually put on the user searching for the data that's stored in the structured database. But with vector databases, the burden is put on you who is actually se…
  [5:30] score=0.597  work properly. Popular implementation of vector databases include Pine Cone and Chroma DB. And these platforms are designed to handle embeddings at scale and pr…
  [0:44] score=0.575  person searching for the data to get the search term formatted correctly. But what if there was a different way to store the data? What if instead of storing th…


## Part 2 — Moment RAG

Now we move the intelligence to the **query side** and make retrieval land on *moments*.

**Ingestion (once):**
1. **Segment into moments** — group cues at semantic breakpoints (cosine distance between adjacent
   cues), so each moment is a coherent span with real `start_ms`/`end_ms`, not an arbitrary window.
2. **Enrich each moment** — one LLM call per moment produces `hypothetical_questions` it answers,
   `keywords`, and a one-line `gist`. (This is HyDE-at-ingestion: we index the *questions a moment
   answers*, so a user's question can match a stored question, not just the raw text.)
3. **Index** — embed moment text **and** the hypothetical questions; build a BM25 sparse index too.

**Query time:** decompose → hybrid retrieve (dense-text + dense-questions + BM25, **RRF-fused**) →
**cross-encoder re-rank** → cited synthesis with timestamp deep-links.

In [6]:
def segment_moments(cues, percentile=80, min_tokens=60, max_tokens=400):
    """Group cues into semantically coherent moments using embedding-distance breakpoints."""
    cue_vecs = embed_texts([c["text"] for c in cues])
    norm = cue_vecs / (np.linalg.norm(cue_vecs, axis=1, keepdims=True) + 1e-9)
    dists = [1 - float(norm[i] @ norm[i + 1]) for i in range(len(cues) - 1)]
    thresh = float(np.percentile(dists, percentile)) if dists else 1.0

    moments, cur, cur_tok = [], [0], len(ENC.encode(cues[0]["text"]))
    for i, d in enumerate(dists):
        nxt_tok = len(ENC.encode(cues[i + 1]["text"]))
        breaking = (d >= thresh and cur_tok >= min_tokens) or cur_tok >= max_tokens
        if breaking:
            moments.append(cur); cur, cur_tok = [i + 1], nxt_tok
        else:
            cur.append(i + 1); cur_tok += nxt_tok
    if cur:
        moments.append(cur)

    out = []
    for idxs in moments:
        window = [cues[j] for j in idxs]
        out.append({
            "moment_id": f"m{len(out):04d}",
            "text": " ".join(c["text"] for c in window),
            "start_ms": window[0]["start_ms"],
            "end_ms": window[-1]["end_ms"],
        })
    return out

moments = segment_moments(cues)
print(f"{len(moments)} moments (vs {len(base_chunks)} fixed chunks)")
for m in moments[:4]:
    print(f'  [{fmt_ts(m["start_ms"])}-{fmt_ts(m["end_ms"])}] {m["text"][:90]}…')

25 moments (vs 13 fixed chunks)
  [0:00-0:23] Let's say your company has an employee handbook that covers policies like time off request…
  [0:21-0:49] In a conventional approach where data is stored in a structured database like SQL, you typ…
  [0:48-1:06] what if there was a different way to store the data? What if instead of storing them by th…
  [1:04-1:24] database returns only relevant data back. This is a spirit of what vector databases try to…


In [7]:
ENRICH_PROMPT = """You are indexing a segment of a YouTube explainer transcript for search.
Return STRICT JSON with exactly these keys:
- "hypothetical_questions": array of 2-4 specific questions a viewer could ask that THIS segment answers.
- "keywords": array of 3-8 salient concepts or entities.
- "gist": one concise sentence summarizing the segment.

Segment:
{text}"""

def enrich_moment(m: dict) -> dict:
    resp = client.chat.completions.create(
        model=LLM_MODEL, temperature=0.1,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": ENRICH_PROMPT.format(text=m["text"])}],
    )
    data = json.loads(resp.choices[0].message.content)
    m["hypothetical_questions"] = list(data.get("hypothetical_questions") or [])[:4]
    m["keywords"] = list(data.get("keywords") or [])
    m["gist"] = (data.get("gist") or "").strip()
    return m

# per-moment calls are independent -> run them concurrently
with ThreadPoolExecutor(max_workers=8) as ex:
    moments = list(ex.map(enrich_moment, moments))

print("enriched. example:")
print(json.dumps({k: moments[0][k] for k in ("gist", "keywords", "hypothetical_questions")}, indent=2))

enriched. example:
{
  "gist": "The segment discusses common employee questions regarding company policies found in the employee handbook, such as dress code and time off requests.",
  "keywords": [
    "employee handbook",
    "time off requests",
    "dress code",
    "equipment use",
    "company policies",
    "database",
    "common questions"
  ],
  "hypothetical_questions": [
    "What is the dress code policy for the office?",
    "How do I request time off during holidays?",
    "Is it allowed to take my company laptop home?"
  ]
}


In [8]:
from rank_bm25 import BM25Okapi

def toks(s: str) -> list[str]:
    return re.findall(r"\w+", s.lower())

# Dense index over moment TEXT
moment_vecs = embed_texts([m["text"] for m in moments])

# Dense index over hypothetical QUESTIONS (each maps back to its moment) — HyDE branch
q_texts, q_owner = [], []
for mi, m in enumerate(moments):
    for q in m["hypothetical_questions"]:
        q_texts.append(q); q_owner.append(mi)
q_vecs = embed_texts(q_texts) if q_texts else np.zeros((0, moment_vecs.shape[1]), dtype=np.float32)

# Sparse BM25 index over moment text
bm25 = BM25Okapi([toks(m["text"]) for m in moments])

print(f"indexed: {len(moments)} moment vectors, {len(q_texts)} question vectors, BM25 ready")

indexed: 25 moment vectors, 98 question vectors, BM25 ready


In [9]:
def decompose(query: str) -> list[str]:
    """One question -> 2-4 retrieval sub-queries (atomic questions pass through)."""
    prompt = (
        "Break the user's question into 2-4 focused sub-queries for retrieval. "
        'Return JSON {"sub_queries": [...]}. If the question is already atomic, '
        "return it as the only element.\n\nQuestion: " + query
    )
    resp = client.chat.completions.create(
        model=LLM_MODEL, temperature=0.1,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}],
    )
    subs = json.loads(resp.choices[0].message.content).get("sub_queries") or [query]
    return [s for s in subs if s][:4] or [query]

def rrf_fuse(ranked_lists: list[list[int]], k: int = 60) -> list[tuple[int, float]]:
    """Reciprocal Rank Fusion across ranked moment-id lists."""
    scores: dict[int, float] = {}
    for lst in ranked_lists:
        for rank, mid in enumerate(lst):
            scores[mid] = scores.get(mid, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: -x[1])

def moment_retrieve(query: str, k: int = 8):
    """Decompose, then per sub-query fuse dense-text + dense-questions + BM25 with RRF."""
    subs = decompose(query)
    ranked_lists = []
    for sq in subs:
        sv = embed_texts([sq])[0]
        ranked_lists.append([mid for mid, _ in cosine_topk(sv, moment_vecs, k)])      # dense text
        if len(q_vecs):                                                                # dense questions
            seen = []
            for qi, _ in cosine_topk(sv, q_vecs, k):
                mo = q_owner[qi]
                if mo not in seen:
                    seen.append(mo)
            ranked_lists.append(seen)
        bm_scores = bm25.get_scores(toks(sq))                                          # sparse BM25
        ranked_lists.append([int(i) for i in np.argsort(-bm_scores)[:k]])
    return subs, rrf_fuse(ranked_lists)

subs, fused = moment_retrieve("How does a vector database index embeddings and search them quickly?")
print("sub-queries:", subs)
print("top fused moments:", [(mid, round(s, 4)) for mid, s in fused[:5]])

sub-queries: ['What is a vector database and how does it work?', 'How are embeddings indexed in a vector database?', 'What techniques are used for fast searching in vector databases?']
top fused moments: [(3, 0.1273), (14, 0.1256), (6, 0.1113), (13, 0.1094), (24, 0.0784)]


In [10]:
USE_RERANK = True
_reranker = None

def get_reranker():
    global _reranker
    if _reranker is None:
        from sentence_transformers import CrossEncoder
        _reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")  # small CPU model
    return _reranker

def rerank(query: str, moment_ids: list[int], top_n: int = 6):
    """Cross-encoder re-scores candidate moments against the ORIGINAL query."""
    rr = get_reranker()
    scores = rr.predict([(query, moments[mid]["text"]) for mid in moment_ids])
    order = np.argsort(-scores)
    return [(moment_ids[i], float(scores[i])) for i in order[:top_n]]

print("reranker stage ready (USE_RERANK =", USE_RERANK, ")")

reranker stage ready (USE_RERANK = True )


In [11]:
def moment_rag(query: str, top_n: int = 6, use_rerank: bool | None = None) -> dict:
    use_rerank = USE_RERANK if use_rerank is None else use_rerank
    subs, fused = moment_retrieve(query)
    cand = [mid for mid, _ in fused[:20]]
    ranked = rerank(query, cand, top_n) if (use_rerank and cand) else fused[:top_n]
    chosen = [moments[mid] for mid, _ in ranked]

    context = "\n\n".join(
        f"[{n+1}] ({fmt_ts(m['start_ms'])}) {m['text']}" for n, m in enumerate(chosen)
    )
    prompt = (
        "Answer the question using ONLY the context below. Cite supporting moments "
        "inline as [n]. If the answer isn't in the context, say so.\n\n"
        f"Context:\n{context}\n\nQuestion: {query}"
    )
    resp = client.chat.completions.create(
        model=LLM_MODEL, temperature=0.2,
        messages=[{"role": "user", "content": prompt}],
    )
    citations = [{
        "n": n + 1, "timestamp": fmt_ts(m["start_ms"]),
        "link": yt_link(m["start_ms"]), "gist": m.get("gist", ""),
    } for n, m in enumerate(chosen)]
    return {"answer": resp.choices[0].message.content,
            "sub_queries": subs, "citations": citations}

res = moment_rag("How does a vector database index embeddings and search them quickly?")
print("Sub-queries:", res["sub_queries"], "\n")
print(res["answer"], "\n\n--- citations ---")
for c in res["citations"]:
    print(f'  [{c["n"]}] {c["timestamp"]}  {c["link"]}\n        {c["gist"]}')

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Sub-queries: ['What is a vector database and how does it work?', 'How are embeddings indexed in a vector database?', 'What techniques are used for fast searching in vector databases?'] 

The context does not provide specific details on how a vector database indexes embeddings and searches them quickly. It mentions that vector databases understand meaning and that they convert sentences into long vectors of numbers for comparison during searches [2][6]. However, it does not elaborate on the indexing process or the mechanisms for quick searching. 

--- citations ---
  [1] 1:04  https://www.youtube.com/watch?v=VVNYQKDLY5s&t=64s
        This segment explains how vector databases allow for searching by meaning rather than value, highlighting the advantages and setup challenges involved.
  [2] 2:41  https://www.youtube.com/watch?v=VVNYQKDLY5s&t=161s
        The segment explains how an embedding model converts sentences into vectors for efficient database searching.
  [3] 5:43  https://www.yo

## Part 3 — Comparison & analysis

We run a shared query set through both pipelines, including a deliberately **multi-faceted**
question that the baseline tends to answer only half of.

In [12]:
QUERIES = [
    "How does a vector database find similar vectors?",
    "What is an embedding?",
    # multi-faceted: exposes the value of query decomposition
    "How does a vector database index embeddings, and how does that differ from a traditional database query?",
]

for q in QUERIES:
    print("=" * 100)
    print("Q:", q)
    b = baseline_rag(q)
    m = moment_rag(q)
    print("\n--- BASELINE ---\n", b["answer"])
    print("\n--- MOMENT RAG ---")
    print("sub-queries:", m["sub_queries"])
    print(m["answer"])
    print("citations:", [f'{c["timestamp"]} -> {c["link"]}' for c in m["citations"]])
    print()

Q: How does a vector database find similar vectors?



--- BASELINE ---
 The context does not provide a specific answer to how a vector database finds similar vectors.

--- MOMENT RAG ---
sub-queries: ['What algorithms do vector databases use to find similar vectors?', 'What is the role of distance metrics in vector similarity search?', 'How do indexing techniques improve the efficiency of vector similarity searches?']
A vector database finds similar vectors by using techniques such as cosine similarity, which enables semantic search. This allows the database to match queries based on meaning rather than exact keyword matches, as demonstrated when a query about wearing jeans to work correctly matches the dress code policy despite not containing the words "dress" or "code" [2]. Additionally, the database employs scoring thresholds to determine how similar the results need to be to be considered a proper match, along with chunk overlap to ensure context is preserved during the retrieval process [1][3].
citations: ['4:59 -> https://www.youtu


--- BASELINE ---
 An embedding is a representation of semantic meanings of words or phrases, stored as a long vector of numbers. It converts values into their meanings, allowing for searches based on meaning rather than exact wording.

--- MOMENT RAG ---
sub-queries: ['What is the definition of an embedding?', 'What are the applications of embeddings in machine learning?', 'How are embeddings created or generated?']
An embedding is a key concept in vector databases that converts sentences into a long vector of numbers, allowing the database to store and retrieve data based on semantic meaning rather than exact wording. It captures the meanings of words and phrases, enabling searches that consider the context and relationships between terms, such as how "holiday" and "vacation" share similar meanings [1][2][3].
citations: ['1:58 -> https://www.youtube.com/watch?v=VVNYQKDLY5s&t=118s', '2:41 -> https://www.youtube.com/watch?v=VVNYQKDLY5s&t=161s', '2:22 -> https://www.youtube.com/watch?v=


--- BASELINE ---
 The context does not provide specific details on how a vector database indexes embeddings. It mentions that vector databases store the meaning of words through embeddings and that they allow searching based on meaning rather than exact keyword matches, which is how traditional SQL databases operate. However, it does not explain the indexing process itself.

--- MOMENT RAG ---
sub-queries: ['How does a vector database index embeddings?', 'What are the differences between vector database queries and traditional database queries?']
A vector database indexes embeddings by converting sentences into long vectors of numbers using an embedding model, which allows for searching based on meaning rather than exact keyword matches. When a document is added to the database, it is processed through this embedding model, and during a search, the database compares the vectors to find relevant results based on semantic similarity [3][4]. 

In contrast, a traditional database query re

In [13]:
# Compact side-by-side summary
def unit_summary():
    rows = [
        ("Retrieval unit", f"{len(base_chunks)} fixed ~{CHUNK_TOKENS}-token chunks",
         f"{len(moments)} semantic moments"),
        ("Boundaries", "arbitrary (token budget)", "topic shifts (semantic breakpoints)"),
        ("Query handling", "single embedding lookup", "decompose -> hybrid (dense+BM25) -> RRF -> rerank"),
        ("Extra index signals", "none", f"{len(q_texts)} hypothetical-question vectors (HyDE)"),
        ("Citations", "chunk text only", "moment + clickable &t= timestamp deep-link"),
    ]
    w = max(len(r[0]) for r in rows)
    print(f'{"":<{w}} | {"BASELINE":<42} | MOMENT RAG')
    print("-" * 110)
    for label, a, b in rows:
        print(f"{label:<{w}} | {a:<42} | {b}")

unit_summary()

                    | BASELINE                                   | MOMENT RAG
--------------------------------------------------------------------------------------------------------------
Retrieval unit      | 13 fixed ~256-token chunks                 | 25 semantic moments
Boundaries          | arbitrary (token budget)                   | topic shifts (semantic breakpoints)
Query handling      | single embedding lookup                    | decompose -> hybrid (dense+BM25) -> RRF -> rerank
Extra index signals | none                                       | 98 hypothetical-question vectors (HyDE)
Citations           | chunk text only                            | moment + clickable &t= timestamp deep-link


### How Moment RAG changes answer quality

- **Citations land on the exact moment.** Every Moment RAG citation carries a real timestamp and a
  `&t=<sec>s` deep-link — you can jump to the spot in the video. Baseline chunks have no reliable,
  topic-aligned timestamp to cite.
- **Decomposition catches multi-part questions.** The multi-faceted query is split into sub-queries,
  so retrieval covers *both* facets (indexing **and** the contrast with traditional queries). The
  baseline's single lookup tends to favor whichever facet is lexically dominant and under-answers
  the other.
- **HyDE-at-ingestion improves matching.** Because each moment is indexed by the *questions it
  answers*, a user's natural-language question can match a stored question even when it shares few
  words with the raw transcript.
- **Hybrid + RRF balances meaning and keywords.** Dense vectors catch paraphrase; BM25 catches exact
  terms (e.g. "HNSW", "cosine"); RRF fuses them so neither dominates.
- **Cross-encoder re-rank sharpens precision.** It re-scores the fused shortlist jointly against the
  query, pushing the truly on-point moment to the top before synthesis.
- **Moments are coherent citeable units.** Semantic boundaries mean a cited moment is a self-contained
  span, not a fragment sliced mid-thought by a token budget.

**Tradeoffs.** Moment RAG costs more at ingestion (one LLM enrichment call per moment) and adds
query-time latency (decomposition + multi-branch retrieval + rerank). For a single video that's
negligible; at scale you'd cache enrichment (the reference commits it) and tune `top_n` / k.

## Appendix — Backing retrieval with a real vector database (ChromaDB)

Everything above uses an in-memory NumPy matrix as an *exact* similarity-search index. Here we
satisfy the **"vector database"** option of the assignment literally: we store the **same Part 1
baseline embeddings** in **ChromaDB** — a real vector DB with an HNSW (approximate-nearest-neighbor)
index and on-disk persistence — and query it.

We reuse the vectors already in `base_matrix`, so indexing adds **no extra OpenAI cost** (only the
one query embedding per question). Chroma returns a **distance** (`cosine distance = 1 − cosine
similarity`), so *lower = closer* — compare it to the cosine *scores* the NumPy baseline printed
in Part 1. On a corpus this small the results match the brute-force version; the difference only
shows up at scale, where Chroma's ANN index avoids comparing the query to every vector.

In [14]:
import chromadb

# In-memory client; swap to chromadb.PersistentClient(path="data/chroma") to persist on disk.
chroma = chromadb.EphemeralClient()
try:
    chroma.delete_collection("baseline_chunks")   # idempotent: wipe on re-run
except Exception:
    pass
collection = chroma.create_collection("baseline_chunks", metadata={"hnsw:space": "cosine"})

# Reuse the embeddings already computed in Part 1 (base_matrix) -> no extra OpenAI cost.
collection.add(
    ids=[c["chunk_id"] for c in base_chunks],
    embeddings=base_matrix.tolist(),
    documents=[c["text"] for c in base_chunks],
    metadatas=[{"start_ms": c["start_ms"], "start": fmt_ts(c["start_ms"])} for c in base_chunks],
)
print(f"stored {collection.count()} chunks in ChromaDB collection 'baseline_chunks'")

stored 13 chunks in ChromaDB collection 'baseline_chunks'


In [15]:
def baseline_rag_chroma(query: str, k: int = 4) -> dict:
    """Same baseline pipeline as Part 1, but retrieval is served by ChromaDB."""
    qv = embed_texts([query])[0].tolist()
    res = collection.query(query_embeddings=[qv], n_results=k)
    docs, metas, dists = res["documents"][0], res["metadatas"][0], res["distances"][0]

    sep = """

"""  # two newlines between context blocks
    context = sep.join(f"[{n+1}] {d}" for n, d in enumerate(docs))
    prompt = f"""Answer the question using ONLY the context below. If the answer isn't in the context, say so.

Context:
{context}

Question: {query}"""
    resp = client.chat.completions.create(
        model=LLM_MODEL, temperature=0.2,
        messages=[{"role": "user", "content": prompt}],
    )
    return {
        "answer": resp.choices[0].message.content,
        "chunks": [{"start": m["start"], "cosine_dist": round(d, 3), "text": doc[:160] + "…"}
                   for m, d, doc in zip(metas, dists, docs)],
    }

out = baseline_rag_chroma("How does a vector database find similar vectors?")
print(out["answer"])
print("--- retrieved from ChromaDB (cosine distance, lower = closer) ---")
for c in out["chunks"]:
    print(f'  [{c["start"]}] dist={c["cosine_dist"]}  {c["text"]}')

The context does not provide a specific answer to how a vector database finds similar vectors.
--- retrieved from ChromaDB (cosine distance, lower = closer) ---
  [3:54] dist=0.351  are working with vector databases and this is the retrieval side. Meaning now that we store the meaning of those words, we have to take on the burden of the ret…
  [1:31] dist=0.394  actually put on the user searching for the data that's stored in the structured database. But with vector databases, the burden is put on you who is actually se…
  [5:30] dist=0.403  work properly. Popular implementation of vector databases include Pine Cone and Chroma DB. And these platforms are designed to handle embeddings at scale and pr…
  [0:44] dist=0.425  person searching for the data to get the search term formatted correctly. But what if there was a different way to store the data? What if instead of storing th…
